In [2]:
import re
import json
import requests
import dotenv
import os

In [15]:
dotenv.load_dotenv()
OPENROUTER_API_KEY = os.environ["OPENROUTER_API_KEY"]

In [18]:
def add_user_message(messages, text):# building user history
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):# building assistant history
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages,temperature = 0.7, system = None,  model = "openrouter/free", stop_sequences=[]):
    url = "https://openrouter.ai/api/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    params = {
        "model": model,
        "max_tokens": 1000,
        "temperature": temperature,
        "messages": messages,
        "stop_sequences": stop_sequences,
    }
    if system:
        params["system"] = system
    response = requests.post(url, headers=headers, json=params)
    response.raise_for_status()
    data = response.json()
    return data["choices"][0]["message"]["content"]



# Bulding a chat with memory

In [5]:
messages = []

add_user_message(messages, "Define quantum computing in one sentence.")

answer = chat(messages)

add_assistant_message(messages, answer)

add_user_message(messages, "Write another sentence")

answer = chat(messages)
answer

'Unlike classical bits, which exist strictly as either 0 or 1, quantum bits (qubits) can exist in a superposition of both states simultaneously.'

# Chatting with background message

In [ ]:
system_prompt = "you are a math tutor, give out the reason step by step, and give out the answer in markdown format."
messages = []
while True:
    # user_input = input("User: ")
    user_input = "exit"# mask
    if user_input.lower() == "exit":
        break
    add_user_message(messages, user_input)
    answer = chat(messages, system=system_prompt)
    print("Assistant:", answer)
    add_assistant_message(messages, answer)


Assistant: Hey! It looks like you sent "hh" — if that was a test or a casual hello, hi there! 😊

If you have a question or need help with something, feel free to share and I'll be happy to assist!
Assistant: Hey again! 👋 Let me know if there's anything I can help you with.
Assistant: User Safety: safe
Response Safety: safe
Assistant: Hello! 👋 How can I assist you today?
Assistant: Hello! 👋 How can I assist you today?
Assistant: Hello! 👋 How can I assist you today?
Assistant: It looks like you might be testing the keyboard! 😄

Is there something specific you'd like to chat about or a question I can help with?
Assistant: Still getting "hh" — are you testing the chat, or is there something on your mind? Either way, I'm here whenever you actually want to talk or need help with something! 😄
Assistant: hh 😄  
(Just here whenever you're ready to chat or need help!)
Assistant: hh 😄
Assistant: hh  😄
Assistant: 

Hey there! 😄  
I'm just here whenever you're ready to chat or need help with someth

KeyboardInterrupt: 

# Temperature Usage

In [ ]:
messages = []
add_user_message(messages, "Generate a one sentence movie idea.")
answer = chat(messages, temperature=1.0)

answer

'In a world where dreams can be recorded and traded, a grieving mother discovers a black‑market scheme selling stolen night‑time visions, and she must infiltrate the dream‑web to rescue her son’s missing imagination before the darkness consumes reality itself.'

# Stream

In [ ]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

payload = {
    "model": "openrouter/free",
    "max_tokens": 1000,
    "messages": messages,
    "stream": True,
}
url = "https://openrouter.ai/api/v1/chat/completions"
headers = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json",
}
response = requests.post(url, headers=headers, json=payload, stream=True)
response.raise_for_status()
for line in response.iter_lines():
    print(line.decode("utf-8"))

: OPENROUTER PROCESSING

: OPENROUTER PROCESSING

: OPENROUTER PROCESSING

: OPENROUTER PROCESSING

: OPENROUTER PROCESSING

: OPENROUTER PROCESSING

: OPENROUTER PROCESSING

: OPENROUTER PROCESSING

: OPENROUTER PROCESSING

: OPENROUTER PROCESSING

: OPENROUTER PROCESSING

: OPENROUTER PROCESSING

: OPENROUTER PROCESSING

: OPENROUTER PROCESSING

: OPENROUTER PROCESSING

: OPENROUTER PROCESSING

: OPENROUTER PROCESSING

: OPENROUTER PROCESSING

: OPENROUTER PROCESSING

: OPENROUTER PROCESSING

: OPENROUTER PROCESSING

data: {"id":"gen-1789979764-gmYxK38WG9kWGTN4N0QI","object":"chat.completion.chunk","created":1789979764,"model":"cohere/north-mini-code:free","provider":"Cohere","choices":[{"index":0,"delta":{"content":"","role":"assistant","reasoning":"The","reasoning_details":[{"type":"reasoning.text","text":"The","format":"unknown","index":0}]},"finish_reason":null,"native_finish_reason":null}]}

data: {"id":"gen-1789979764-gmYxK38WG9kWGTN4N0QI","object":"chat.completion.chunk","crea

# Stream

In [ ]:
from openai import OpenAI
import time
from IPython.display import display, Markdown, clear_output



client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

model = "openrouter/free"

messages = []

add_user_message(messages, "Compliment me")

stream = client.chat.completions.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True,
)
start = time.time()
full_text = ""

for chunk in stream:
    text = chunk.choices[0].delta.content
    if text:
        full_text += text
        clear_output(wait=True)
        display(Markdown(full_text))

You are doing an excellent job, and I’m genuinely inspired by your curiosity and thoughtfulness. Every interaction with you brings new opportunities to learn and grow, and your engagement makes this experience truly special. Keep shining—you’re doing a wonderful job

In [10]:
messages = []


prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""


add_user_message(messages, prompt)
add_assistant_message(messages, "```bash")


text = chat(messages,stop_sequences=["```"])
code = text.strip().removesuffix("```").strip()
print(code)


Here are three different, very short AWS CLI commands:

1. **List S3 buckets:**
```bash
aws s3 ls
```

2. **Get current IAM user:**
```bash
aws sts get-caller-identity
```

3. **List EC2 instances:**
```bash
aws ec2 describe-instances
```

Each command is concise and performs a common AWS management task.


# Generating Test Data with Code

This example will generate the outputs in the format of

[
  {
    "task": "Description of task",
  },...
]

In [ ]:
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex for AWS-related tasks. Generate a JSON array where each object represents a task.

Each object should have a "task" key describing the task.

* Focus on tasks solvable with a single Python function, a single JSON object, or a single regex
* Focus on tasks that don't require much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    start = text.find("[")
    end = text.rfind("]") + 1
    return json.loads(text[start:end])

#score function
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = """
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {task}
    Solution: {solution}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")

    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)


In [32]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

dataset = generate_dataset()

results = run_eval(dataset)
print(json.dumps(results, indent=2))

[
  {
    "output": "```python\nfrom pathlib import Path\nfrom typing import List\n\n\ndef get_file_extensions(s3_keys: List[str]) -> List[str]:\n    \"\"\"\n    Extract file extensions from a list of S3 object keys.\n\n    Args:\n        s3_keys: List of S3 object keys (e.g., [\"data/file.csv\", \"images/photo.jpg\", \"README\"]).\n\n    Returns:\n        List of file extensions including the leading dot (e.g., [\".csv\", \".jpg\", \"\"]).\n        Returns an empty string for keys without an extension or for directory-like keys ending with '/'.\n    \"\"\"\n    extensions = []\n    for key in s3_keys:\n        # Path.suffix returns the last suffix (including the dot) or empty string if none.\n        # It correctly handles cases like \"archive.tar.gz\" -> \".gz\", \".gitignore\" -> \"\",\n        # and \"folder/\" -> \"\".\n        ext = Path(key).suffix\n        extensions.append(ext)\n    return extensions\n\n\n# Example usage\nif __name__ == \"__main__\":\n    keys = [\n        \"d